# IITM Reinforcement Learning Course Project
## Industrial Inventory Control using Reinforcement Learning

This starter notebook demonstrates how to:

1. generate the assigned parameter variant from the official roll number;
2. create and inspect the Gymnasium environment;
3. convert between order quantities and the internal `MultiDiscrete` action;
4. interact with the environment for one episode;
5. organise experiments for five distinct RL techniques.

The notebook intentionally does **not** provide complete RL algorithms or a complete multi-seed evaluator. Those are part of the project work.

## 1. Setup

Keep the `industrial_inventory_env` folder in the same project directory as this notebook. Install the approved packages using:

```bash
pip install -r requirements.txt
```

In [1]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from industrial_inventory_env import (
    IndustrialInventoryEnv,
    generate_student_config,
    public_config_summary,
)

np.set_printoptions(suppress=True)
print("Imports completed.")

Imports completed.


## 2. Generate the assigned parameter variant

Enter the official roll number below. The same normalized roll number always generates the same variant for this project version. Do not use another student's roll number and do not modify the generation utility.

In [2]:
ROLL_NUMBER = "ENTER_YOUR_ROLL_NUMBER"  # Replace this text once.

student_config = generate_student_config(ROLL_NUMBER)
config_summary = public_config_summary(student_config)

print("Assigned configuration generated successfully.")
for key, value in config_summary.items():
    print(f"{key}: {value}")

ValueError: Enter your official roll number before generating the configuration.

Record the generated `variant_id` and `config_fingerprint` in the notebook and brief report. The leaderboard evaluator will use separate common hidden configurations; it will not use student-supplied parameter values.

## 3. Create the environment

`scenario_mode="random"` permits stationary episodes and combinations of the declared seasonal, trend and temporary-shock characteristics. Episode parameters are generated deterministically from the reset seed.

In [ ]:
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)

observation, info = env.reset(seed=2026)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("Variant:", info["variant_id"])
print("Scenario components:", info["episode_parameters"]["scenario_components"])
print("Episode demand multipliers:", info["episode_parameters"]["demand_multipliers"])
print("Episode initial inventory:", info["episode_parameters"]["initial_inventory"])
print("Episode delay probabilities:", info["episode_parameters"]["delay_probabilities"])

In [ ]:
for key, value in observation.items():
    print(f"{key:22s} shape={value.shape}, dtype={value.dtype}")
    print(value)
    print()

## 4. Action representation

The environment accepts internal indices `[0, ..., 10]` for each product. The leaderboard `run_policy(observation)` function must return actual quantities from `{0, 10, ..., 100}`. Use the supplied conversion utility during local interaction.

In [ ]:
order_quantities = [40, 20, 0]
action_indices = env.quantities_to_action_indices(order_quantities)

print("Actual order quantities:", order_quantities)
print("Internal action indices:", action_indices.tolist())
print("Converted back:", env.action_indices_to_quantities(action_indices).tolist())

## 5. One-step interaction example

At the beginning of the decision step, the policy receives the current observation and selects the order quantities. The environment then processes arrivals, capacity, the new order, demand and cost before returning the next observation.

In [ ]:
next_observation, reward, terminated, truncated, step_info = env.step(action_indices)

print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Demand:", step_info["demand"])
print("Daily costs:", step_info["costs"])
print("Next inventory:", next_observation["inventory"])

## 6. Demonstration policy

The following heuristic is included only to demonstrate the interaction loop. It is **not** one of the required RL techniques and should not be presented as an RL submission.

In [ ]:
def demonstration_policy(observation: dict) -> list[int]:
    """Simple deterministic inventory-position heuristic for demonstration."""
    inventory = np.asarray(observation["inventory"], dtype=float)
    pipeline = np.asarray(observation["arrival_pipeline"], dtype=float)
    inventory_position = inventory + pipeline.sum(axis=1)

    targets = np.asarray([110.0, 100.0, 120.0])
    required = np.maximum(targets - inventory_position, 0.0)
    quantities = np.clip(np.ceil(required / 10.0) * 10.0, 0.0, 100.0)
    return quantities.astype(int).tolist()

## 7. Run one deterministic episode

This example shows the Gymnasium loop and cost collection for a single seed. Students must build their own local evaluation process for comparing techniques across multiple seeds and scenario families.

In [ ]:
demo_env = IndustrialInventoryEnv(
    student_config,
    scenario_mode="random",
    domain_randomization=True,
)
observation, reset_info = demo_env.reset(seed=101)

records = []
while True:
    quantities = demonstration_policy(observation)
    action = demo_env.quantities_to_action_indices(quantities)
    observation, reward, terminated, truncated, info = demo_env.step(action)

    records.append(
        {
            "day": info["day"],
            "reward": reward,
            "daily_cost": info["costs"]["daily_total"],
            "episode_cost": info["costs"]["episode_total"],
            "inventory_p1": int(observation["inventory"][0]),
            "inventory_p2": int(observation["inventory"][1]),
            "inventory_p3": int(observation["inventory"][2]),
            "order_p1": int(quantities[0]),
            "order_p2": int(quantities[1]),
            "order_p3": int(quantities[2]),
        }
    )

    if terminated or truncated:
        break

one_episode_results = pd.DataFrame(records)
print("Episode days:", len(one_episode_results))
print("Total episode cost:", one_episode_results["daily_cost"].sum())
one_episode_results.head()

In [ ]:
one_episode_results.plot(
    x="day",
    y=["inventory_p1", "inventory_p2", "inventory_p3"],
    figsize=(10, 4),
    title="Inventory during the demonstration episode",
)
plt.ylabel("Units")
plt.show()

## 8. Build the local evaluation loop

Create a function that evaluates a supplied policy across multiple disclosed validation seeds and multiple declared scenario families. At minimum, record:

- average and standard deviation of total episode cost;
- holding, stockout, ordering and discarding costs;
- service level or unfulfilled demand;
- computational time;
- the seeds and scenario modes used.

Do not tune against one favourable seed. Keep the final public and private leaderboard episodes hidden and evaluator-controlled.

In [ ]:
# TODO: Implement your own evaluation function.
# Suggested signature:
#
# def evaluate_policy(policy, seeds, scenario_modes):
#     ...
#     return results_dataframe
#
# The function should call policy(observation), convert quantities to action indices,
# execute complete 50-day episodes and aggregate unscaled episode cost.


## 9. Track the five distinct techniques

Maintain a compact experiment table throughout the project. Use the exact technique categories announced by the instructors.

In [ ]:
technique_tracker = pd.DataFrame(
    {
        "technique": ["Technique 1", "Technique 2", "Technique 3", "Technique 4", "Technique 5"],
        "main_state_representation": ["", "", "", "", ""],
        "important_hyperparameters": ["", "", "", "", ""],
        "local_average_cost": [np.nan] * 5,
        "public_submission_id": ["", "", "", "", ""],
        "public_average_cost": [np.nan] * 5,
        "policy_file": ["", "", "", "", ""],
        "model_artifact": ["", "", "", "", ""],
    }
)
technique_tracker

## 10. Submission checks

For every policy file:

```bash
python policy_validation_tests.py path/to/policy_file.py
```

The final notebook must reproduce the training/export process and clearly map each frozen leaderboard submission to its technique, policy file and model artefact. The brief report should focus on what was implemented, what results were obtained and what was inferred.